# BAA10Y Adaptive Agent Test


In [13]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from BAA10Y_forecasting.adaptive_agent.tuner import (
    BAA10YTuner,
)
import pandas as pd
from IPython.display import display
from BAA10Y_forecasting.predictors.benchmark_configs import (
    NOTEBOOK_01_BENCHMARKS,
    get_benchmark_config,
)
import logging

logging.getLogger(
    "LiteLLM"
).setLevel(
    logging.CRITICAL
)

logging.getLogger(
    "litellm"
).setLevel(
    logging.CRITICAL
)

In [ ]:
def find_repo_root() -> Path:
    """Find the agentic-forecasting repository root."""

    here = Path.cwd().resolve()

    for candidate in (
        here,
        *here.parents,
    ):
        if (
            (
                candidate
                / "pyproject.toml"
            ).exists()
            and (
                candidate
                / "aieng-forecasting"
            ).is_dir()
        ):
            return candidate

    raise RuntimeError(
        "Could not find repository root."
    )


CHECKS = []


def add_check(
    category: str,
    name: str,
    passed: bool,
    details: str = "",
) -> None:
    """Add one result to the notebook health report."""

    CHECKS.append({
        "category": category,
        "check": name,
        "status": (
            "PASS"
            if passed
            else "NOT YET"
        ),
        "details": details,
    })


ROOT = find_repo_root()

# This directory separates the Notebook 04 LightGBM
# demonstration from older smoke-test and tuning state.
#
# To start a completely new study later, DEMO_RUN_ID
DEMO_RUN_ID = (
    "lightgbm_h5_default_plus_hyoas_no_gold_v1"
)

STATE_PATH = (
    ROOT
    / "implementations"
    / "BAA10Y_forecasting"
    / "adaptive_agent"
    / "state"
    / DEMO_RUN_ID
    / "tuning_state.yaml"
)


print("\nAdaptive LightGBM state:")
print(STATE_PATH)


Adaptive LightGBM state:
/home/coder/agentic-forecasting/implementations/BAA10Y_forecasting/adaptive_agent/state/lightgbm_h5_default_no_gold_v1/tuning_state.yaml


## 1. Register and verify adaptive-agent tools

In [15]:
from BAA10Y_forecasting.adaptive_agent.agent import (
    build_baa10y_adaptive_config,
)
from BAA10Y_forecasting.adaptive_agent.skill_state import (
    TuningStateStore,
)
from BAA10Y_forecasting.adaptive_agent.skill_tools import (
    build_baa10y_tuning_tools,
)
from BAA10Y_forecasting.adaptive_agent.tuner import (
    BAA10YTuner,
)


add_check(
    "technical",
    "Adaptive modules import",
    True,
)

TOOLS = build_baa10y_tuning_tools(
    state_path=STATE_PATH,
)

TOOL_BY_NAME = {
    tool.__name__: tool
    for tool in TOOLS
}

EXPECTED_TOOLS = {
    "get_tuning_state",
    "list_tuning_candidates",
    "get_search_space",
    "run_tuning_trial",
    "run_adaptive_search",
    "get_search_diagnostics",
    "freeze_search_candidate",
    "run_frozen_validation",
    "compare_tuning_trials",
    "promote_tuning_candidate",
    "reject_tuning_candidate",
}

tool_names = set(
    TOOL_BY_NAME
)

missing_tools = (
    EXPECTED_TOOLS
    - tool_names
)

tools_ok = not missing_tools

add_check(
    "technical",
    "Adaptive tuning tools registered",
    tools_ok,
    str(sorted(tool_names)),
)

print(
    "Registered adaptive-agent tools:"
)

for name in sorted(tool_names):
    print(" -", name)

assert tools_ok, (
    "Missing adaptive-agent tools: "
    f"{sorted(missing_tools)}"
)

Registered adaptive-agent tools:
 - compare_tuning_trials
 - freeze_search_candidate
 - get_search_diagnostics
 - get_search_space
 - get_tuning_state
 - list_tuning_candidates
 - promote_tuning_candidate
 - reject_tuning_candidate
 - run_adaptive_search
 - run_frozen_validation
 - run_tuning_trial


## 2. Configure the LightGBM adaptive test


In [ ]:
import json

from BAA10Y_forecasting.predictors.benchmark_configs import (
    get_benchmark_config,
)


def call_json_tool(
    tool_name: str,
    **kwargs,
) -> dict:
    """Call one registered agent tool and parse its JSON response."""

    if tool_name not in TOOL_BY_NAME:
        raise KeyError(
            f"Unknown tool {tool_name!r}. "
            f"Available tools: "
            f"{sorted(TOOL_BY_NAME)}"
        )

    raw_output = (
        TOOL_BY_NAME[tool_name](
            **kwargs
        )
    )

    payload = json.loads(
        raw_output
    )

    if payload.get("status") == "error":
        raise RuntimeError(
            payload.get(
                "message",
                payload,
            )
        )

    return payload


METHOD = "lightgbm"
HORIZON = 1
COVARIATE_PANEL = "default_plus_hyoas"

INITIAL_TRIALS = 6
TOTAL_TRIALS = 12


BENCHMARK_PARAMETERS = (
    get_benchmark_config(
        METHOD,
        COVARIATE_PANEL,
    )
)

SEARCH_SPACE = call_json_tool(
    "get_search_space",
    method=METHOD,
)


search_space_ok = (
    SEARCH_SPACE["method"]
    == METHOD
    and SEARCH_SPACE["strategy"]
    == "optuna_tpe"
    and COVARIATE_PANEL
    in SEARCH_SPACE[
        "covariate_panels"
    ]
)

add_check(
    "technical",
    "LightGBM search space available",
    search_space_ok,
    (
        f"strategy="
        f"{SEARCH_SPACE['strategy']}; "
        f"panel={COVARIATE_PANEL}"
    ),
)

assert search_space_ok


print("Adaptive test configuration:")
print(" Method:", METHOD)
print(" Horizon:", HORIZON)
print(
    " Covariate panel:",
    COVARIATE_PANEL,
)
print(
    " Initial trials:",
    INITIAL_TRIALS,
)
print(
    " Total trials:",
    TOTAL_TRIALS,
)

print("\nMatching baseline parameters:")
display(BENCHMARK_PARAMETERS)

print("\nApproved LightGBM search space:")
display(
    SEARCH_SPACE[
        "parameter_space"
    ]
)

print("\nAllowed agent focus actions:")
display(
    SEARCH_SPACE[
        "focus_actions"
    ]
)

Adaptive test configuration:
 Method: lightgbm
 Horizon: 5
 Covariate panel: default
 Initial trials: 5
 Total trials: 10

Matching baseline parameters:


{'lags': 5,
 'lags_past_covariates': 5,
 'num_samples': 100,
 'lgbm_kwargs': {'num_threads': 1,
  'n_jobs': 1,
  'verbosity': -1,
  'random_state': 42}}


Approved LightGBM search space:


{'lags': [3, 5, 10, 21],
 'lags_past_covariates': [3, 5, 10, 21],
 'n_estimators': {'minimum': 50, 'maximum': 400, 'step': 50},
 'learning_rate': {'minimum': 0.02, 'maximum': 0.15, 'scale': 'log'},
 'tree_shapes': [{'max_depth': 3, 'num_leaves': 7},
  {'max_depth': 4, 'num_leaves': 15},
  {'max_depth': 6, 'num_leaves': 31},
  {'max_depth': -1, 'num_leaves': 15},
  {'max_depth': -1, 'num_leaves': 31}],
 'min_child_samples': [10, 20, 40],
 'reg_alpha_l1': {'minimum': 0.0, 'maximum': 2.0},
 'reg_lambda_l2': {'minimum': 0.0, 'maximum': 5.0},
 'subsample': [0.7, 0.85, 1.0],
 'colsample_bytree': [0.7, 0.85, 1.0]}


Allowed agent focus actions:


['broad_search',
 'regularize_more',
 'reduce_complexity',
 'stabilize_boosting',
 'continue_tpe']

In [17]:
import yaml

from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
)


SPEC_DIRECTORY = (
    ROOT
    / "implementations"
    / "BAA10Y_forecasting"
    / "specs"
)

EXPERIMENT_SPECS = {
    "development": (
        "baa10y_tune_development_2025.yaml"
    ),
    "inner_validation": (
        "baa10y_tune_inner_validation_2025.yaml"
    ),
    "outer_validation": (
        "baa10y_validate_2025.yaml"
    ),
}


def load_backtest_spec(
    filename: str,
) -> MultiTargetBacktestSpec:
    """Load and validate one backtest specification."""

    path = (
        SPEC_DIRECTORY
        / filename
    )

    with path.open(
        encoding="utf-8"
    ) as file:
        contents = yaml.safe_load(
            file
        )

    return (
        MultiTargetBacktestSpec
        .model_validate(contents)
    )


LOADED_SPECS = {
    name: load_backtest_spec(
        filename
    )
    for name, filename
    in EXPERIMENT_SPECS.items()
}


period_rows = []

for name, specification in (
    LOADED_SPECS.items()
):
    start = pd.Timestamp(
        specification.start
    )

    end = pd.Timestamp(
        specification.end
    )

    origins = pd.date_range(
        start=start,
        end=end,
        freq="B",
    )[::specification.stride]

    period_rows.append({
        "period": name,
        "start": start.date(),
        "end": end.date(),
        "stride": specification.stride,
        "number_of_origins": (
            len(origins)
        ),
        "first_origin": (
            origins.min().date()
        ),
        "last_origin": (
            origins.max().date()
        ),
    })


EVALUATION_PERIODS = pd.DataFrame(
    period_rows
)

display(EVALUATION_PERIODS)


development_end = pd.Timestamp(
    LOADED_SPECS[
        "development"
    ].end
)

inner_validation_start = pd.Timestamp(
    LOADED_SPECS[
        "inner_validation"
    ].start
)

inner_validation_end = pd.Timestamp(
    LOADED_SPECS[
        "inner_validation"
    ].end
)

outer_validation_start = pd.Timestamp(
    LOADED_SPECS[
        "outer_validation"
    ].start
)


periods_separated = (
    development_end
    < inner_validation_start
    and inner_validation_end
    < outer_validation_start
)

add_check(
    "governance",
    "Tuning and validation periods separated",
    periods_separated,
    (
        f"development_end="
        f"{development_end.date()}; "
        f"inner_start="
        f"{inner_validation_start.date()}; "
        f"inner_end="
        f"{inner_validation_end.date()}; "
        f"outer_start="
        f"{outer_validation_start.date()}"
    ),
)

assert periods_separated, (
    "Development, inner-validation and "
    "outer-validation periods overlap."
)

print(
    "PASS: The three evaluation periods "
    "are separated."
)

,period,start,end,stride,number_of_origins,first_origin,last_origin
0,development,2025-01-08,2025-04-02,5,13,2025-01-08,2025-04-02
1,inner_validation,2025-05-07,2025-08-06,5,14,2025-05-07,2025-08-06
2,outer_validation,2025-09-10,2025-12-17,5,15,2025-09-10,2025-12-17


PASS: The three evaluation periods are separated.


Define the LightGBM agent assignment

In [18]:
LIVE_AGENT_PROMPT = f"""
Tune {METHOD} for the {HORIZON}-business-day BAA10Y forecast using
the {COVARIATE_PANEL} covariate panel and its matching fixed benchmark.

Run {INITIAL_TRIALS} broad trials, review paired diagnostics, choose
an evidence-based focus action, and continue to {TOTAL_TRIALS} total
trials. Freeze only a robust, non-overfitting candidate that improves
both development and inner-validation CRPS.

Do not access outer validation, eval_2026, or promote a model.
""".strip()


print(LIVE_AGENT_PROMPT)

Tune lightgbm for the 5-business-day BAA10Y forecast using
the default covariate panel and its matching fixed benchmark.

Run 5 broad trials, review paired diagnostics, choose
an evidence-based focus action, and continue to 10 total
trials. Freeze only a robust, non-overfitting candidate that improves
both development and inner-validation CRPS.

Do not access outer validation, eval_2026, or promote a model.


In [19]:
async def run_live_agent_with_trace(
    prompt: str,
):
    """Run the adaptive agent and record every tool call."""

    from aieng.forecasting.methods.agentic import (
        build_adk_agent,
    )
    from google.adk.runners import (
        InMemoryRunner,
    )
    from google.genai import (
        types as genai_types,
    )

    agent_config = (
        build_baa10y_adaptive_config(
            state_path=STATE_PATH,
            max_output_tokens=4096,
        )
    )

    live_agent = build_adk_agent(
        agent_config
    )

    app_name = (
        "baa10y_lightgbm_adaptive_test"
    )

    user_id = "notebook04_user"

    runner = InMemoryRunner(
        agent=live_agent,
        app_name=app_name,
    )

    session = await (
        runner.session_service
        .create_session(
            app_name=app_name,
            user_id=user_id,
        )
    )

    message = genai_types.Content(
        role="user",
        parts=[
            genai_types.Part(
                text=prompt
            ),
        ],
    )

    tool_trace = []
    response_parts = []

    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=message,
        ):
            content = getattr(
                event,
                "content",
                None,
            )

            parts = (
                getattr(
                    content,
                    "parts",
                    None,
                )
                or []
            )

            for part in parts:
                function_call = getattr(
                    part,
                    "function_call",
                    None,
                )

                if function_call is not None:
                    arguments = (
                        getattr(
                            function_call,
                            "args",
                            None,
                        )
                        or {}
                    )

                    tool_trace.append({
                        "tool": (
                            function_call.name
                        ),
                        "arguments": dict(
                            arguments
                        ),
                    })

                text_value = getattr(
                    part,
                    "text",
                    None,
                )

                if (
                    event.is_final_response()
                    and text_value
                ):
                    response_parts.append(
                        text_value
                    )

    finally:
        await runner.close()

    return (
        "\n".join(response_parts),
        tool_trace,
    )


(
    LIVE_AGENT_REPLY,
    LIVE_AGENT_TRACE,
) = await run_live_agent_with_trace(
    LIVE_AGENT_PROMPT
)


print("Agent response:")
print()

print(
    LIVE_AGENT_REPLY
    or "No final text response returned."
)

print(
    "\nNumber of tool calls:",
    len(LIVE_AGENT_TRACE),
)

Agent response:

### Observed Metrics

* **Method**: `lightgbm`
* **Horizon**: 5 business days
* **Covariate Panel**: `default`
* **Search Strategy**: `optuna_tpe`
* **Completed Trial Count**: 10 (plus 1 baseline trial)
* **Matching Benchmark CRPS**:
  * **Development**: 4.3689
  * **Inner-Validation**: 2.9526
* **Best Candidate (`lightgbm_default_768b502cac6b`, Trial 0) CRPS**:
  * **Development**: 4.2488 (2.75% improvement over benchmark)
  * **Inner-Validation**: 2.7972 (5.26% improvement over benchmark)
* **Generalization Gap (Best Candidate)**: -34.17% (compared to -32.42% for benchmark)
* **Possible-Overfitting Status**: False
* **Robust-Candidate Status**: True

---

### Agent Interpretation

* **Tuning History**: 
  1. An initial **`broad_search`** was performed for the first 5 trials.
  2. Overfitting tendencies were observed in complex parameter settings (e.g., Trial 2 and Trial 3, which combined deeper trees, `num_leaves=31`, and small `min_child_samples=10`). These configur

In [23]:
LEARNING_PROCEDURE_PROMPT = """
Explain the adaptive learning procedure for the existing study:

baa10y_lightgbm_default_h5_tune_paired_2025

Call only get_tuning_state and get_search_diagnostics. Do not run
additional trials, validation, freezing, rejection or promotion.

Present:
1. The benchmark.
2. Trials 0-4 from broad_search.
3. Evidence leading to reduce_complexity.
4. Trials 5-9 from the focused search.
5. Parameter changes that helped or hurt.
6. Why the finalist was robust and frozen.
7. Current independent-validation status.

Separate Optuna proposals, agent decisions, backtester metrics and
persistent state. Include a trial-by-trial table. Do not claim that
the LLM updated its weights.
""".strip()


(
    LEARNING_PROCEDURE_REPLY,
    LEARNING_PROCEDURE_TRACE,
) = await run_live_agent_with_trace(
    LEARNING_PROCEDURE_PROMPT
)

print(
    LEARNING_PROCEDURE_REPLY
)

display(
    pd.DataFrame(
        LEARNING_PROCEDURE_TRACE
    )
)

### Overview of the Adaptive Learning Procedure

The study **`baa10y_lightgbm_default_h5_tune_paired_2025`** is an adaptive hyperparameter tuning study for the LightGBM method over a 5-business-day horizon (`h5`) using the `default` covariate panel. 

The adaptive tuning was executed in two phases:
1. **`broad_search`** (Trials 0–4): An initial exploration of the registered hyperparameter search space using Optuna TPE.
2. **`reduce_complexity`** (Trials 5–9): A focused search round triggered by the agent after observing overfitting patterns in complex tree configurations.

Below is a detailed breakdown of the trials, the agent


,tool,arguments
0,get_tuning_state,{}
1,get_search_diagnostics,{'study_name': 'baa10y_lightgbm_default_h5_tun...


In [20]:
STATE_RESULT = call_json_tool(
    "get_tuning_state"
)

STUDY_NAME = (
    "baa10y_lightgbm_"
    "default_h5_"
    "tune_paired_2025"
)

STUDY_RECORD = next(
    study
    for study in STATE_RESULT["studies"]
    if study["study_name"]
    == STUDY_NAME
)

print(
    "Frozen:",
    STUDY_RECORD["search_frozen"],
)

print(
    "Candidate:",
    STUDY_RECORD[
        "best_candidate_id"
    ],
)

print(
    "Validation decision:",
    STUDY_RECORD[
        "validation_decision"
    ],
)

assert (
    STUDY_RECORD["search_frozen"]
    is True
)

assert (
    STUDY_RECORD[
        "best_candidate_id"
    ]
    == "lightgbm_default_768b502cac6b"
)

Frozen: True
Candidate: lightgbm_default_768b502cac6b
Validation decision: pending_validation


In [21]:
VALIDATION_RESULT = call_json_tool(
    "run_frozen_validation",
    study_name=STUDY_NAME,
    force_refresh=False,
)

display(VALIDATION_RESULT)

{'status': 'success',
 'study_name': 'baa10y_lightgbm_default_h5_tune_paired_2025',
 'method': 'lightgbm',
 'horizon': 5,
 'covariate_panel': 'default',
 'experiment': 'validate_2025',
 'required_repeats': 1,
 'baseline': {'candidate_id': 'lightgbm_default_ef942949340b',
  'mean_crps': 2.4603708909150517,
  'crps_std': 0.0,
  'scores': [2.4603708909150517],
  'parameters': {'lags': 5,
   'lags_past_covariates': 5,
   'num_samples': 100,
   'lgbm_kwargs': {'num_threads': 1,
    'n_jobs': 1,
    'verbosity': -1,
    'random_state': 42}}},
 'candidate': {'candidate_id': 'lightgbm_default_768b502cac6b',
  'mean_crps': 2.5288223812852335,
  'crps_std': 0.0,
  'scores': [2.5288223812852335],
  'parameters': {'lags': 5,
   'lags_past_covariates': 21,
   'num_samples': 100,
   'lgbm_kwargs': {'num_threads': 1,
    'n_jobs': 1,
    'verbosity': -1,
    'random_state': 42,
    'n_estimators': 150,
    'learning_rate': 0.05757368425309993,
    'max_depth': -1,
    'num_leaves': 15,
    'min_child

In [ ]:
STATE_RESULT = call_json_tool(
    "get_tuning_state"
)

STUDY_NAME = (
    "baa10y_lightgbm_"
    "default_h5_"
    "tune_paired_2025"
)

STUDY_RECORD = next(
    study
    for study in STATE_RESULT["studies"]
    if study["study_name"]
    == STUDY_NAME
)

print(
    "Frozen:",
    STUDY_RECORD["search_frozen"],
)

print(
    "Candidate:",
    STUDY_RECORD[
        "best_candidate_id"
    ],
)

print(
    "Validation decision:",
    STUDY_RECORD[
        "validation_decision"
    ],
)

assert (
    STUDY_RECORD["search_frozen"]
    is True
)

assert (
    STUDY_RECORD[
        "best_candidate_id"
    ]
    == "lightgbm_default_768b502cac6b"
)

In [24]:
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

from aieng.forecasting.evaluation import (
    EvalTracker,
    MultiTargetEvalSpec,
    multi_evaluate,
    save_multi_eval_results,
)
from aieng.forecasting.methods import (
    DartsLightGBMPredictor,
    LastValuePredictor,
)

from BAA10Y_forecasting import (
    DEFAULT_COVARIATE_SERIES_IDS,
    build_baa10y_multivariate_service,
)
from BAA10Y_forecasting.leaderboard import (
    build_leaderboard,
)
from BAA10Y_forecasting.predictors.adaptive_candidates import (
    build_adaptive_predictor,
)
from BAA10Y_forecasting.predictors.benchmark_configs import (
    get_benchmark_config,
)


def find_repository_root(
    start: Path,
) -> Path:
    """Find the repository directory containing implementations/."""

    resolved_start = start.resolve()

    for candidate in (
        resolved_start,
        *resolved_start.parents,
    ):
        implementation_dir = (
            candidate
            / "implementations"
            / "BAA10Y_forecasting"
        )

        if implementation_dir.is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find the repository root."
    )


REPO_ROOT = find_repository_root(
    Path.cwd()
)

BAA10Y_DIR = (
    REPO_ROOT
    / "implementations"
    / "BAA10Y_forecasting"
)

SPEC_PATH = (
    BAA10Y_DIR
    / "specs"
    / "baa10y_eval_2026.yaml"
)

STATE_PATH = (
    BAA10Y_DIR
    / "adaptive_agent"
    / "state"
    / "tuning_state.yaml"
)

EVAL_TRACKER_PATH = (
    STATE_PATH.with_name(
        "eval_2026_runs.yaml"
    )
)

EVAL_ARTIFACT_DIR = (
    REPO_ROOT
    / "data"
    / "predictions"
    / "baa10y_adaptive_eval_2026"
)

print("Repository:", REPO_ROOT)
print("Evaluation specification:", SPEC_PATH)
print("Evaluation tracker:", EVAL_TRACKER_PATH)

Repository: /home/coder/agentic-forecasting
Evaluation specification: /home/coder/agentic-forecasting/implementations/BAA10Y_forecasting/specs/baa10y_eval_2026.yaml
Evaluation tracker: /home/coder/agentic-forecasting/implementations/BAA10Y_forecasting/adaptive_agent/state/eval_2026_runs.yaml


In [26]:
assert (
    VALIDATION_RESULT["status"]
    == "success"
), VALIDATION_RESULT

FINAL_STUDY_NAME = (
    VALIDATION_RESULT["study_name"]
)

TUNING_STATE = call_json_tool(
    "get_tuning_state"
)

assert (
    TUNING_STATE["status"]
    == "success"
), TUNING_STATE

FINAL_STUDY = next(
    study
    for study in TUNING_STATE["studies"]
    if (
        study["study_name"]
        == FINAL_STUDY_NAME
    )
)

assert FINAL_STUDY["search_frozen"]
assert FINAL_STUDY["best_candidate_id"]
assert FINAL_STUDY["best_parameters"]

print(
    "Frozen candidate:",
    FINAL_STUDY["best_candidate_id"],
)

print(
    "Independent-validation decision:",
    VALIDATION_RESULT["decision"],
)

print(
    "Independent-validation improvement:",
    f"{VALIDATION_RESULT['improvement_pct']:.2f}%",
)

print(
    "Purpose of eval_2026:",
    "diagnostic comparison only",
)

if (
    VALIDATION_RESULT["decision"]
    != "promote_tuned"
):
    print(
        "NOTE: This candidate was not promoted. "
        "Its eval_2026 result will not change "
        "the retain-baseline decision."
    )

Frozen candidate: lightgbm_default_768b502cac6b
Independent-validation decision: retain_baseline
Independent-validation improvement: -2.78%
Purpose of eval_2026: diagnostic comparison only
NOTE: This candidate was not promoted. Its eval_2026 result will not change the retain-baseline decision.


In [27]:
with SPEC_PATH.open(
    encoding="utf-8"
) as file:
    FULL_EVAL_SPEC = (
        MultiTargetEvalSpec.model_validate(
            yaml.safe_load(file)
        )
    )

HORIZON = 5
TASK_ID = "baa10y_change_5b"

HORIZON_TASKS = [
    task
    for task in FULL_EVAL_SPEC.tasks
    if HORIZON in task.horizons
]

assert len(HORIZON_TASKS) == 1
assert (
    HORIZON_TASKS[0].task_id
    == TASK_ID
)

# Keep the original spec_id so EvalTracker enforces
# the max_runs value from baa10y_eval_2026.yaml.
EVAL_SPEC_H5 = (
    FULL_EVAL_SPEC.model_copy(
        update={
            "tasks": HORIZON_TASKS,
        }
    )
)

print("Spec ID:", EVAL_SPEC_H5.spec_id)
print("Start:", EVAL_SPEC_H5.start.date())
print("End:", EVAL_SPEC_H5.end.date())
print("Forecast horizon:", HORIZON)
print("Maximum protected runs:", EVAL_SPEC_H5.max_runs)

Spec ID: baa10y_eval_2026
Start: 2026-02-03
End: 2026-03-24
Forecast horizon: 5
Maximum protected runs: 5


In [28]:
EVAL_SERVICE = (
    build_baa10y_multivariate_service(
        windows=(1, 5, 21),
        include_covariates=True,
        covariate_series_ids=(
            DEFAULT_COVARIATE_SERIES_IDS
        ),
        start="2016-01-01",
        refresh=False,
    )
)

registered_series = set(
    EVAL_SERVICE.series_ids
)

missing_covariates = [
    series_id
    for series_id
    in DEFAULT_COVARIATE_SERIES_IDS
    if series_id
    not in registered_series
]

assert not missing_covariates, (
    "Missing default covariates: "
    f"{missing_covariates}"
)

EVAL_COVARIATES = list(
    DEFAULT_COVARIATE_SERIES_IDS
)

print(
    "Default covariates:",
    len(EVAL_COVARIATES),
)

print(EVAL_COVARIATES)

Default covariates: 10
['vix_level_l1b', 'vix_log_ret_1b_l1b', 'ust10y_level_l1b', 'ust2y10y_spread_l1b', 'fed_funds_level_l1b', 'cpi_mom_logdiff_l1b', 'unemployment_rate_l1b', 'oil_log_ret_1b_l1b', 'dollar_index_log_ret_1b_l1b', 'nasdaq_log_ret_1b_l1b']


In [29]:
NOTEBOOK01_NAIVE = (
    LastValuePredictor()
)

NOTEBOOK01_LIGHTGBM_PARAMETERS = (
    get_benchmark_config(
        "lightgbm",
        "default",
    )
)

NOTEBOOK01_LIGHTGBM = (
    DartsLightGBMPredictor(
        covariate_series_ids=(
            EVAL_COVARIATES
        ),
        **NOTEBOOK01_LIGHTGBM_PARAMETERS,
    )
)

ADAPTIVE_CANDIDATE = {
    "candidate_id": (
        FINAL_STUDY[
            "best_candidate_id"
        ]
    ),
    "method": "lightgbm",
    "description": (
        "Frozen adaptive-agent LightGBM finalist"
    ),
    "params": (
        FINAL_STUDY[
            "best_parameters"
        ]
    ),
    "covariate_panel": (
        FINAL_STUDY[
            "covariate_panel"
        ]
    ),
    "allowed_horizons": [HORIZON],
    "is_baseline": False,
}

ADAPTIVE_LIGHTGBM = (
    build_adaptive_predictor(
        candidate=ADAPTIVE_CANDIDATE,
        covariate_series_ids=(
            EVAL_COVARIATES
        ),
    )
)

FINALISTS = [
    NOTEBOOK01_NAIVE,
    NOTEBOOK01_LIGHTGBM,
    ADAPTIVE_LIGHTGBM,
]

FINALIST_LABELS = {
    NOTEBOOK01_NAIVE.predictor_id: (
        "Notebook 01 — Last Value"
    ),
    NOTEBOOK01_LIGHTGBM.predictor_id: (
        "Notebook 01 — LightGBM + covariates"
    ),
    ADAPTIVE_LIGHTGBM.predictor_id: (
        "Adaptive agent — LightGBM + covariates"
    ),
}

FINALIST_COVARIATES = {
    NOTEBOOK01_NAIVE.predictor_id: [],
    NOTEBOOK01_LIGHTGBM.predictor_id: (
        EVAL_COVARIATES
    ),
    ADAPTIVE_LIGHTGBM.predictor_id: (
        EVAL_COVARIATES
    ),
}

for predictor in FINALISTS:
    print(
        FINALIST_LABELS[
            predictor.predictor_id
        ],
        "->",
        predictor.predictor_id,
    )

Notebook 01 — Last Value -> last_value_naive
Notebook 01 — LightGBM + covariates -> darts_lightgbm_cov
Adaptive agent — LightGBM + covariates -> baa10y_lightgbm_77cf6afe97


In [30]:
assert (
    ADAPTIVE_CANDIDATE["candidate_id"]
    == "lightgbm_default_768b502cac6b"
)

assert (
    ADAPTIVE_CANDIDATE["params"]
    == FINAL_STUDY["best_parameters"]
)

print(
    "Optuna candidate ID:",
    ADAPTIVE_CANDIDATE["candidate_id"],
)

print(
    "Evaluation predictor ID:",
    ADAPTIVE_LIGHTGBM.predictor_id,
)

print(
    "Parameters identical:",
    ADAPTIVE_CANDIDATE["params"]
    == FINAL_STUDY["best_parameters"],
)

Optuna candidate ID: lightgbm_default_768b502cac6b
Evaluation predictor ID: baa10y_lightgbm_77cf6afe97
Parameters identical: True


In [31]:
def flatten_parameters(
    parameters: dict,
) -> dict:
    """Flatten predictor and LightGBM parameters for comparison."""

    flattened = {}

    for name, value in parameters.items():
        if name == "lgbm_kwargs":
            flattened.update(value)
        else:
            flattened[name] = value

    return flattened


baseline_flat = flatten_parameters(
    NOTEBOOK01_LIGHTGBM_PARAMETERS
)

adaptive_flat = flatten_parameters(
    FINAL_STUDY["best_parameters"]
)

parameter_names = sorted(
    set(baseline_flat)
    | set(adaptive_flat)
)

PARAMETER_COMPARISON = pd.DataFrame(
    [
        {
            "parameter": parameter,
            "notebook_01_benchmark": (
                baseline_flat.get(
                    parameter,
                    "default",
                )
            ),
            "adaptive_agent": (
                adaptive_flat.get(
                    parameter,
                    "default",
                )
            ),
            "changed": (
                baseline_flat.get(
                    parameter,
                    "default",
                )
                != adaptive_flat.get(
                    parameter,
                    "default",
                )
            ),
        }
        for parameter in parameter_names
    ]
)

display(
    PARAMETER_COMPARISON.sort_values(
        [
            "changed",
            "parameter",
        ],
        ascending=[
            False,
            True,
        ],
    )
)

,parameter,notebook_01_benchmark,adaptive_agent,changed
0,colsample_bytree,default,1.000000,True
2,lags_past_covariates,5,21.000000,True
3,learning_rate,default,0.057574,True
4,max_depth,default,-1.000000,True
5,min_child_samples,default,40.000000,True
6,n_estimators,default,150.000000,True
8,num_leaves,default,15.000000,True
12,reg_alpha,default,0.278988,True
13,reg_lambda,default,1.460723,True
14,subsample,default,0.700000,True


In [32]:
#Set RUN_PROTECTED_EVAL = True only when you are ready
RUN_PROTECTED_EVAL = True

EVAL_TRACKER = EvalTracker(
    EVAL_TRACKER_PATH
)

runs_used = EVAL_TRACKER.runs_for(
    EVAL_SPEC_H5.spec_id
)

required_runs = len(FINALISTS)

runs_remaining = (
    EVAL_SPEC_H5.max_runs
    - runs_used
)

print("Protected runs already used:", runs_used)
print("Runs required now:", required_runs)
print("Runs remaining before execution:", runs_remaining)

assert required_runs <= runs_remaining, (
    "Not enough protected-evaluation budget. "
    f"Required={required_runs}, "
    f"remaining={runs_remaining}."
)

if not RUN_PROTECTED_EVAL:
    print(
        "Protected evaluation has not run. "
        "Change RUN_PROTECTED_EVAL to True once."
    )
else:
    if (
        "FINAL_EVAL_RESULTS"
        not in globals()
    ):
        FINAL_EVAL_RESULTS = {}

    for predictor in FINALISTS:
        predictor_id = (
            predictor.predictor_id
        )

        if (
            predictor_id
            in FINAL_EVAL_RESULTS
        ):
            print(
                "Reusing in-memory result:",
                FINALIST_LABELS[
                    predictor_id
                ],
            )
            continue

        print(
            "Evaluating:",
            FINALIST_LABELS[
                predictor_id
            ],
        )

        predictor_results = (
            multi_evaluate(
                predictor=predictor,
                spec=EVAL_SPEC_H5,
                data_service=EVAL_SERVICE,
                tracker=EVAL_TRACKER,
            )
        )

        FINAL_EVAL_RESULTS[
            predictor_id
        ] = predictor_results

        artifact_paths = (
            save_multi_eval_results(
                results=predictor_results,
                spec=EVAL_SPEC_H5,
                store_dir=(
                    EVAL_ARTIFACT_DIR
                ),
            )
        )

        print(
            "Saved:",
            artifact_paths,
        )

    print(
        "Protected runs used after evaluation:",
        EVAL_TRACKER.runs_for(
            EVAL_SPEC_H5.spec_id
        ),
    )

Protected runs already used: 0
Runs required now: 3
Runs remaining before execution: 5
Evaluating: Notebook 01 — Last Value
Saved: {'baa10y_change_5b': PosixPath('/home/coder/agentic-forecasting/data/predictions/baa10y_adaptive_eval_2026/baa10y_eval_2026/last_value_naive__baa10y_change_5b__eval_run1.yaml')}
Evaluating: Notebook 01 — LightGBM + covariates
Saved: {'baa10y_change_5b': PosixPath('/home/coder/agentic-forecasting/data/predictions/baa10y_adaptive_eval_2026/baa10y_eval_2026/darts_lightgbm_cov__baa10y_change_5b__eval_run2.yaml')}
Evaluating: Adaptive agent — LightGBM + covariates


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7895c31b1250>


Saved: {'baa10y_change_5b': PosixPath('/home/coder/agentic-forecasting/data/predictions/baa10y_adaptive_eval_2026/baa10y_eval_2026/baa10y_lightgbm_77cf6afe97__baa10y_change_5b__eval_run3.yaml')}
Protected runs used after evaluation: 3


In [33]:
assert "FINAL_EVAL_RESULTS" in globals(), (
    "Run the protected-evaluation cell first."
)

assert (
    len(FINAL_EVAL_RESULTS)
    == len(FINALISTS)
), (
    "Not every finalist completed evaluation."
)

FINAL_EVAL_DF = build_leaderboard(
    FINAL_EVAL_RESULTS,
    EVAL_SERVICE,
    covariates_by_predictor=(
        FINALIST_COVARIATES
    ),
    labels_by_predictor=(
        FINALIST_LABELS
    ),
)

FINAL_EVAL_DF["rank"] = (
    FINAL_EVAL_DF[
        "mean_crps"
    ]
    .rank(
        method="min",
        ascending=True,
    )
    .astype(int)
)

FINAL_EVAL_DF = (
    FINAL_EVAL_DF.sort_values(
        "mean_crps"
    )
    .reset_index(drop=True)
)

display(
    FINAL_EVAL_DF[
        [
            "rank",
            "model",
            "horizon",
            "mean_crps",
            "n_predictions",
            "skipped_origins",
            "dir_accuracy",
            "run_number",
        ]
    ].style.format(
        {
            "mean_crps": "{:.4f}",
            "dir_accuracy": "{:.2%}",
        }
    )
)

,rank,model,horizon,mean_crps,n_predictions,skipped_origins,dir_accuracy,run_number
0,1,Notebook 01 — LightGBM + covariates,5,3.6418,8,0,25.00%,2
1,2,Adaptive agent — LightGBM + covariates,5,3.7473,8,0,25.00%,3
2,3,Notebook 01 — Last Value,5,6.5000,8,0,50.00%,1


In [34]:
adaptive_row = (
    FINAL_EVAL_DF[
        FINAL_EVAL_DF["model"]
        == (
            "Adaptive agent — "
            "LightGBM + covariates"
        )
    ]
    .iloc[0]
)

notebook01_rows = (
    FINAL_EVAL_DF[
        FINAL_EVAL_DF["model"]
        .str.startswith(
            "Notebook 01"
        )
    ]
)

best_notebook01_row = (
    notebook01_rows.sort_values(
        "mean_crps"
    )
    .iloc[0]
)

adaptive_improvement_pct = (
    100.0
    * (
        best_notebook01_row[
            "mean_crps"
        ]
        - adaptive_row[
            "mean_crps"
        ]
    )
    / best_notebook01_row[
        "mean_crps"
    ]
)

if adaptive_improvement_pct > 0:
    final_result = (
        "ADAPTIVE MODEL WINS"
    )
else:
    final_result = (
        "NOTEBOOK 01 MODEL WINS"
    )

FINAL_COMPARISON = pd.DataFrame(
    [
        {
            "evaluation_window": (
                "eval_2026"
            ),
            "horizon": HORIZON,
            "best_notebook_01_model": (
                best_notebook01_row[
                    "model"
                ]
            ),
            "best_notebook_01_crps": (
                best_notebook01_row[
                    "mean_crps"
                ]
            ),
            "adaptive_model": (
                adaptive_row["model"]
            ),
            "adaptive_crps": (
                adaptive_row[
                    "mean_crps"
                ]
            ),
            "adaptive_improvement_pct": (
                adaptive_improvement_pct
            ),
            "result": final_result,
        }
    ]
)

display(
    FINAL_COMPARISON.style.format(
        {
            "best_notebook_01_crps": (
                "{:.4f}"
            ),
            "adaptive_crps": "{:.4f}",
            "adaptive_improvement_pct": (
                "{:.2f}%"
            ),
        }
    )
)

print(final_result)

,evaluation_window,horizon,best_notebook_01_model,best_notebook_01_crps,adaptive_model,adaptive_crps,adaptive_improvement_pct,result
0,eval_2026,5,Notebook 01 — LightGBM + covariates,3.6418,Adaptive agent — LightGBM + covariates,3.7473,-2.90%,NOTEBOOK 01 MODEL WINS


NOTEBOOK 01 MODEL WINS
